# Module 2 In-Class Lab: Language Models, Tokens, and Embeddings

In this lab, we move from **text generation** to the internal representations
that make language models useful: tokens and embeddings. We will inspect four
levels of representation:

1. token IDs used by a generative language model;
2. contextual token embeddings produced by an encoder model;
3. sentence embeddings used to compare complete texts; and
4. song embeddings learned from playlist co-occurrence.

## Learning objectives

By the end of the lab, you should be able to:

- explain why a token is not necessarily a word;
- interpret the shape of a contextual embedding tensor;
- distinguish token, word, sentence, and item embeddings;
- use cosine similarity to compare embeddings; and
- explain how co-occurrence can turn playlists into training data.

**Suggested workflow:** run one section at a time and answer each checkpoint
before moving forward. Cells that download pretrained models require internet
access the first time they are run.


## 0. Environment setup

The Phi-3 example is the only part of the notebook that strongly benefits from
a GPU. Loading a multi-billion-parameter model on a laptop can be very slow and
may exhaust memory, so the notebook automatically skips that section when CUDA
is unavailable. The smaller encoder and sentence-embedding models can run on a
CPU, although their first download may take a few minutes.

If the imports below fail in a new environment, uncomment and run the install
command, restart the kernel, and then begin again from this section.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# (Local Env Users) Run once in a new environment, then restart the kernel.
# %pip install -q "transformers>=4.41,<5" "sentence-transformers>=3,<6" \
#  "gensim>=4.3.3,<5" "accelerate>=0.31" "numpy>=1.26,<2" \
#  "scipy>=1.11,<1.13" pandas torch sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 68.7 MB/s eta 0:00:00
  Installing build dependencies ... canceled
ERROR: Operation cancelled by user


In [ ]:
from pathlib import Path
import html as html_library
import random

import numpy as np
import pandas as pd
import torch
from IPython.display import HTML, display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

HAS_CUDA = torch.cuda.is_available()
HAS_MPS = bool(
    hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
)
RUNTIME_DEVICE = "cuda" if HAS_CUDA else ("mps" if HAS_MPS else "cpu")

# Remote model/tokenizer cells can be disabled for a fully offline walkthrough.
RUN_REMOTE_MODELS = True

# Phi-3 is large enough that we only load it automatically on a CUDA GPU.
RUN_LARGE_LLM = HAS_CUDA

print(f"PyTorch device: {RUNTIME_DEVICE}")
print(f"Run remote model examples: {RUN_REMOTE_MODELS}")
print(f"Run Phi-3 generation examples: {RUN_LARGE_LLM}")


PyTorch device: cuda
Run remote model examples: True
Run Phi-3 generation examples: True


# Part I. From a prompt to generated text

A **tokenizer** converts text to integer token IDs. A **language model** converts
those IDs into probability distributions over possible next tokens. The model
generates text by repeatedly choosing a next token and appending it to the
sequence.

The tokenizer and model must come from compatible checkpoints. Token ID 100 in
one vocabulary can mean something entirely different in another vocabulary.
We therefore load both from `microsoft/Phi-3-mini-4k-instruct`.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

LLM_CHECKPOINT = "microsoft/Phi-3-mini-4k-instruct"
llm = None
llm_tokenizer = None

if RUN_LARGE_LLM:
    llm_tokenizer = AutoTokenizer.from_pretrained(LLM_CHECKPOINT)
    llm = AutoModelForCausalLM.from_pretrained(
        LLM_CHECKPOINT,
        device_map="cuda",
        torch_dtype="auto",
        trust_remote_code=False,
    )
    print(f"Loaded {LLM_CHECKPOINT} on {llm.device}.")
else:
    print(
        "Phi-3 was not loaded because a CUDA GPU was not detected. "
        "Use a GPU runtime and rerun this cell to enable the generation demos."
    )


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Loaded microsoft/Phi-3-mini-4k-instruct on cuda:0.


### 1.1 A high-level generation pipeline

An instruction-tuned model expects special markers that identify user and
assistant turns. `apply_chat_template` inserts the correct markers for this
checkpoint. The `pipeline` then handles tokenization, generation, and decoding.

We use greedy decoding (`do_sample=False`), so the output is reproducible for a
fixed model version. Sampling would make the answer more varied but less
predictable.


In [ ]:
if RUN_LARGE_LLM:
    generator = pipeline(
        "text-generation",
        model=llm,
        tokenizer=llm_tokenizer,
        return_full_text=False,
    )

    joke_messages = [
        {"role": "user", "content": "Create a funny joke about chickens."}
    ]
    joke_prompt = llm_tokenizer.apply_chat_template(
        joke_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    joke_outputs = generator(
        joke_prompt,
        max_new_tokens=80,
        do_sample=True,
        temperature=1.1,
        top_p=0.95,
        num_return_sequences=3,
    )

    for number, output in enumerate(joke_outputs, start=1):
        print(f"Joke {number}:")
        print(output["generated_text"])
        print()
else:
    print("Generation skipped; set RUN_LARGE_LLM=True in a CUDA runtime.")

[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Joke 1:
Why did the chicken join the circus? Because it wanted to be a spectacle!

Joke 2:
Why do chickens love to swim? Because it's the only time they can wear their feathers in a trend!

Joke 3:
Why did the chicken join the dance team? Because it heard it was the hottest item on the "barnyard bounce!"



### 1.2 The same process without the pipeline

The next cell exposes the steps hidden by the pipeline. It creates a chat-formatted
prompt, tokenizes it, sends the resulting tensors to the model's device, and
decodes only the newly generated tokens. Slicing away the input tokens prevents
the prompt from being printed as if it were part of the answer.


In [ ]:
if RUN_LARGE_LLM:
    email_messages = [
        {
            "role": "user",
            "content": (
                "Write an email apologizing to Sarah for the tragic gardening "
                "mishap. Explain how it happened."
            ),
        }
    ]
    email_prompt = llm_tokenizer.apply_chat_template(
        email_messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    email_inputs = llm_tokenizer(
        email_prompt,
        return_tensors="pt",
    ).to(llm.device)

    with torch.inference_mode():
        email_output_ids = llm.generate(
            **email_inputs,
            max_new_tokens=350,
            do_sample=True,
            temperature=1.1,
            top_p=0.95,
            num_return_sequences=1,
            pad_token_id=llm_tokenizer.eos_token_id,
        )

    prompt_length = email_inputs["input_ids"].shape[1]

    for number, output_ids in enumerate(email_output_ids, start=1):
        new_token_ids = output_ids[prompt_length:]
        generated_email = llm_tokenizer.decode(
            new_token_ids,
            skip_special_tokens=True,
        )

        print(f"Email {number}:")
        print(generated_email)
        print()
else:
    print(
        "Direct generation skipped; "
        "set RUN_LARGE_LLM=True in a CUDA runtime."
    )

Email 1:
Subject: Apology for the Unfortunate Gardening Incident

Dear Sarah,

I hope this email finds you in good health and spirits. I am writing this email with a heavy heart to apologize to you for the unfortunate accident that occurred while tending to your garden earlier today.

As you know, I have been helping you with gardening for quite some time now and have always tried my best to ensure everything goes smoothly. However, today's event was truly out of my control, and for that, I feel deeply sorry.

The mishap happened when we were planting your new rose bushes. As I was carefully placing each one in its designated hole, I failed to notice a small, hidden rock just beneath the surface. In an attempt to avoid disturbing the earth, I accidentally hit the rock with the spade, causing it to fly off in an unpredictable manner. Unfortunately, it landed squarely on top of your beloved tulip bed, causing considerable damage to your beautiful flowers.

Please understand that this inc

### 1.3 Inspecting token IDs

Token IDs are vocabulary lookup keys, not measurements. A larger ID does not
represent a “larger” or more important token. The table below shows four views
of each position: its sequence position, integer ID, tokenizer vocabulary symbol,
and decoded text fragment.


In [ ]:
if RUN_LARGE_LLM:
    prompt_ids = email_inputs["input_ids"][0].detach().cpu().tolist()
    token_table = pd.DataFrame(
        {
            "position": range(len(prompt_ids)),
            "token_id": prompt_ids,
            "vocabulary_token": llm_tokenizer.convert_ids_to_tokens(prompt_ids),
            "decoded_piece": [
                repr(llm_tokenizer.decode([token_id]))
                for token_id in prompt_ids
            ],
        }
    )
    display(token_table)
else:
    print("Token-ID inspection skipped with the Phi-3 section.")

,position,token_id,vocabulary_token,decoded_piece
0,0,32010,<|user|>,'<|user|>'
1,1,14350,▁Write,'Write'
2,2,385,▁an,'an'
3,3,4876,▁email,'email'
4,4,27746,▁apolog,'apolog'
5,5,5281,izing,'izing'
6,6,304,▁to,'to'
7,7,19235,▁Sarah,'Sarah'
8,8,363,▁for,'for'
9,9,278,▁the,'the'


In [ ]:
# These IDs reproduce the source notebook's small token-composition example.
if RUN_LARGE_LLM:
    for token_ids in ([3323], [622], [3323, 622], [29901]):
        decoded = llm_tokenizer.decode(token_ids)
        print(f"{token_ids!s:>12} -> {decoded!r}")
else:
    print("Token-composition example skipped with the Phi-3 section.")

       [385] -> 'an'
       [622] -> 'ject'
 [3323, 622] -> 'Subject'
     [29901] -> ':'


**Checkpoint 1**

1. Which work is performed by the tokenizer, and which by the model?
2. Why is the chat template safer than manually typing strings such as `<|assistant|>`?
3. What would you expect to change if `do_sample=True`?

# Part II. Comparing tokenizers

Tokenization is a learned design choice. Different tokenizers may split the same
input differently because they use different vocabularies, training corpora,
casing rules, and algorithms. This affects sequence length, cost, multilingual
coverage, and how easily a model can represent code or unusual symbols.

The deliberately awkward test string below contains capitalization, Unicode,
programming syntax, repeated spaces, and arithmetic. Each colored block is one
token; hover over it to see the integer token ID.


In [ ]:
TOKEN_COLORS = [
    "#66c2a5",
    "#fc8d62",
    "#8da0cb",
    "#e78ac3",
    "#a6d854",
    "#ffd92f",
]

def show_tokens(text, tokenizer_name):
    # Display tokenizer boundaries while making spaces and newlines visible.
    comparison_tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    token_ids = comparison_tokenizer(text, add_special_tokens=False).input_ids

    blocks = []
    for position, token_id in enumerate(token_ids):
        piece = comparison_tokenizer.decode([token_id])
        visible_piece = (
            piece.replace(" ", "␠")
            .replace("\t", "⇥")
            .replace("\n", "↵\n")
        )
        escaped_piece = html_library.escape(visible_piece) or "∅"
        color = TOKEN_COLORS[position % len(TOKEN_COLORS)]
        blocks.append(
            f'<span title="token id: {token_id}" '
            f'style="background:{color}; color:#111; padding:3px 5px; '
            f'margin:2px; border-radius:4px; display:inline-block;">'
            f'{escaped_piece}</span>'
        )

    display(HTML("".join(blocks)))
    print(f"{tokenizer_name}: {len(token_ids)} tokens")
    return token_ids


tokenizer_test_text = '''
English and CAPITALIZATION
🎵 鸟
show_tokens False None elif == >= else: two tabs:"    " Three tabs: "       "
12.0*50=600
'''.strip()


In [ ]:
tokenizer_checkpoints = [
    "bert-base-uncased",
    "bert-base-cased",
    "gpt2",
    "google/flan-t5-small",
    "Xenova/gpt-4",  # Hugging Face representation of the GPT-4 tokenizer
    "facebook/galactica-1.3b",
    "microsoft/Phi-3-mini-4k-instruct",
]

tokenizer_results = {}
if RUN_REMOTE_MODELS:
    for checkpoint in tokenizer_checkpoints:
        print(f"\n{checkpoint}")
        try:
            tokenizer_results[checkpoint] = show_tokens(
                tokenizer_test_text, checkpoint
            )
        except Exception as error:
            # A failed or restricted download should not stop the remaining comparisons.
            print(f"Unavailable: {type(error).__name__}: {error}")

    print(
        "\nbigcode/starcoder2-15b is intentionally skipped because the source "
        "notebook notes that access may need to be requested first."
    )
else:
    print("Tokenizer downloads skipped because RUN_REMOTE_MODELS=False.")



bert-base-uncased


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.14M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/3.00 [00:00<?, ?B/s]

facebook/galactica-1.3b: 54 tokens

microsoft/Phi-3-mini-4k-instruct


microsoft/Phi-3-mini-4k-instruct: 52 tokens

bigcode/starcoder2-15b is intentionally skipped because the source notebook notes that access may need to be requested first.


**Checkpoint 2 — compare with a partner**

- Which tokenizer uses the fewest tokens? Do not assume that it is universally
  “best”; describe one tradeoff.
- How do the cased and uncased BERT tokenizers treat `CAPITALIZATION`?
- Which characters or code fragments are split in surprising ways?
- Add one domain-specific string from your own work. What could poor tokenization
  mean for a model trained on that domain?


# Part III. Contextual token embeddings

After tokenization, a transformer maps each token position to a dense vector.
These vectors are **contextual**: the vector for a word such as “bank” can change
depending on whether the surrounding text discusses money or a river.

We use one checkpoint for both the tokenizer and model. The source notebook used
two different DeBERTa checkpoints, which can silently pair incompatible
vocabularies and weights.


In [ ]:
from transformers import AutoModel

ENCODER_CHECKPOINT = "microsoft/deberta-v3-xsmall"
encoder_tokenizer = None
encoder_model = None

if RUN_REMOTE_MODELS:
    try:
        encoder_tokenizer = AutoTokenizer.from_pretrained(ENCODER_CHECKPOINT)
        encoder_model = AutoModel.from_pretrained(ENCODER_CHECKPOINT).to(
            RUNTIME_DEVICE
        )
        encoder_model.eval()
        print(f"Loaded {ENCODER_CHECKPOINT} on {RUNTIME_DEVICE}.")
    except Exception as error:
        print(f"Encoder unavailable: {type(error).__name__}: {error}")
else:
    print("Contextual encoder download skipped.")


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  241MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from d

Loaded microsoft/deberta-v3-xsmall on cuda.


In [ ]:
if encoder_model is not None:
    encoder_inputs = encoder_tokenizer("Hello world", return_tensors="pt")
    encoder_inputs = {
        name: tensor.to(RUNTIME_DEVICE)
        for name, tensor in encoder_inputs.items()
    }

    with torch.inference_mode():
        token_embeddings = encoder_model(**encoder_inputs).last_hidden_state

    token_embeddings = token_embeddings.detach().cpu()
    encoder_ids = encoder_inputs["input_ids"][0].detach().cpu().tolist()
    encoder_tokens = encoder_tokenizer.convert_ids_to_tokens(encoder_ids)

    print(f"Embedding tensor shape: {tuple(token_embeddings.shape)}")
    display(
        pd.DataFrame(
            token_embeddings[0, :, :6].numpy(),
            index=encoder_tokens,
            columns=[f"dimension_{i}" for i in range(6)],
        )
    )
else:
    print("Contextual embedding example skipped because the encoder is unavailable.")


Embedding tensor shape: (1, 4, 384)


/usr/local/lib/python3.13/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,dimension_0,dimension_1,dimension_2,dimension_3,dimension_4,dimension_5
[CLS],-3.306641,-0.050751,-0.109680,-0.000236,-0.143921,-0.303467
▁Hello,0.892090,0.073975,-0.160522,2.181641,-0.484863,-0.663086
▁world,0.084717,0.637207,-0.306152,1.539062,-0.253662,-0.560059
[SEP],-3.162109,-0.142822,-0.094788,0.227905,-0.225220,-0.350342


The embedding tensor has shape `(1, 4, 384)`:

- `1` means that one text was processed;
- `4` means that the text was represented by four token positions, including
  special tokens; and
- `384` means that each token is represented by an embedding containing 384
  values.

More generally, transformer documentation describes this shape as:

`(batch size, sequence length, hidden size)`

Here, **hidden size** is simply the model's name for the number of dimensions in
each contextual token embedding. For this model, the hidden size is 384.

The table displays only the first six of those 384 dimensions so that the output
is easier to inspect. Individual dimensions do not have simple labels such as
“positivity” or “importance.” Meaning is distributed across the complete
384-dimensional vector.

**Checkpoint 3:** Why are there four token embeddings even though `Hello world`
contains only two words? Identify the special tokens in the table.

# Part IV. Sentence embeddings

A token encoder produces one vector per token position. Many search, clustering,
and retrieval tasks instead need **one vector per text**. A sentence-transformer
model pools token-level information into a single vector designed so that
semantically related texts lie near one another.

Because the embeddings below are normalized, their dot product equals cosine
similarity. Values closer to 1 indicate more similar directions.


In [ ]:
from sentence_transformers import SentenceTransformer

SENTENCE_CHECKPOINT = "sentence-transformers/all-mpnet-base-v2"
sentence_model = None

if RUN_REMOTE_MODELS:
    try:
        sentence_model = SentenceTransformer(
            SENTENCE_CHECKPOINT, device=RUNTIME_DEVICE
        )
        print(f"Loaded {SENTENCE_CHECKPOINT} on {RUNTIME_DEVICE}.")
    except Exception as error:
        print(f"Sentence model unavailable: {type(error).__name__}: {error}")
else:
    print("Sentence-embedding model download skipped.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded sentence-transformers/all-mpnet-base-v2 on cuda.


In [ ]:
sentences = [
    "Best movie ever!",
    "That film was excellent.",
    "The forecast calls for rain tomorrow.",
]

if sentence_model is not None:
    sentence_embeddings = sentence_model.encode(
        sentences,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    similarity_matrix = sentence_embeddings @ sentence_embeddings.T

    print(f"Sentence embedding matrix shape: {sentence_embeddings.shape}")
    display(
        pd.DataFrame(
            similarity_matrix,
            index=sentences,
            columns=sentences,
        ).round(3)
    )
else:
    print("Sentence similarity example skipped because the model is unavailable.")


Sentence embedding matrix shape: (3, 768)


,Best movie ever!,That film was excellent.,The forecast calls for rain tomorrow.
Best movie ever!,1.000,0.567,0.028
That film was excellent.,0.567,1.000,0.068
The forecast calls for rain tomorrow.,0.028,0.068,1.000


**Checkpoint 4**

1. Which off-diagonal pair is most similar? Is that what you expected?
2. Replace the unrelated weather sentence with a sarcastic movie review. Does the
   model recognize the intended meaning?
3. Why is the diagonal exactly (or extremely close to) 1?

Remember that cosine similarity is evidence from a representation model, not a
guarantee that two statements are factually equivalent.


# Part V. Your Turn: Learning song embeddings from playlists

Embeddings are not limited to language. If songs that appear near one another in
playlists tend to be related, we can treat each playlist like a “sentence” and
each song ID like a “word.” Word2Vec then learns song vectors from local
co-occurrence patterns.

Take-Home: Implement `Word Embeddings Beyond LLMs` from the textbook-provided notebook: https://github.com/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter02/Chapter%202%20-%20Tokens%20and%20Token%20Embeddings.ipynb

# Wrap-up: one word, four kinds of representation

| Representation | Unit represented | Depends on context? | Learned from |
|---|---|---:|---|
| Token ID | Vocabulary symbol | No | Tokenizer vocabulary |
| Contextual token embedding | Token at one position | Yes | Transformer training |
| Sentence embedding | Complete text | Yes | Similarity-oriented model training |
| Playlist song embedding | Song ID | Yes, during training | Playlist co-occurrence |
